# NextEra Energy Coding Quiz
## Author: Teja Dasari
Date: 1/12/2025

### Question 1
The data in table_1a.csv and table_1b.csv refers to an operational wind farm.

a. What is the total power in kW for operational turbines? That is, if all operational turbines were operating
at their rated power, what would the total power of the wind farm be? What about non-operational
turbines, what is the total power that is not operational?

b. How many turbines of each unique turbine model (based on rated power) are operational?

c. In a single plot, display the turbine locations, their rated capacity and operational status.


### Solution:

In [2]:
# imports
import pandas as pd
import numpy as np

In [8]:
# Reading the data from CSV files

df_1a = pd.read_csv('CodingQuestions_NEA_AP_table_1a.csv')
df_1b = pd.read_csv('CodingQuestions_NEA_AP_table_1b.csv')
 
# Checkout the data in the dataframes
print(df_1a.head(2))
print(df_1b.head(2))

   turbine        lat         lon  rated_power_kw
0        2  32.653019 -101.891187            2300
1        5  32.535653 -101.875779            2300
  turbine_id  operational
0       T013         True
1       T014         True


In [9]:
# Table Clean-up
"""
Some values are inconsistent and needs clean up  
"""
# Clean the data in the tables
df_1b.turbine_id = df_1b.turbine_id.apply(lambda id:id.strip("_")) # Removes the '_' from the turbine ids

# Sort Turbines as per the ID
df_1a_srtd = df_1a.sort_values(by='turbine')
df_1b_srtd = df_1b.sort_values(by='turbine_id')

# Add a column named 'turbine' to table_1b contaning turbine number
df_1b_srtd['turbine'] = pd.to_numeric(df_1b_srtd.turbine_id.str.split("T").str[1])
 
# Merge tables into one table
wnd_frm = pd.merge(df_1a_srtd, df_1b_srtd, on='turbine')


### Question 1.a
What is the total power in kW for operational turbines? That is, if all operational turbines were operating
at their rated power, what would the total power of the wind farm be? What about non-operational
turbines, what is the total power that is not operational?

In [17]:
# Solution for Question 1.a

Tot_Pwr_Opr = wnd_frm[wnd_frm.operational]['rated_power_kw'].sum()
print('Total power in kW for operational turbines = {} kW '. format(Tot_Pwr_Opr))

Tot_Pwr_NonOpr = wnd_frm[wnd_frm.operational==False]['rated_power_kw'].sum()
print('Total power in kW for non-operational turbines = {} kW '. format(Tot_Pwr_NonOpr))

"""
b. How many turbines of each unique turbine model (based on rated power) are operational?
"""
#wnd_frm_rp = wnd_frm.groupby(['rated_power_kw','operational'])['turbine'].count()
wnd_frm_rp = wnd_frm.groupby(['rated_power_kw','operational'])['turbine'].count().reset_index(name='count')
#wnd_frm_rp = pd.DataFrame(wnd_frm_rp)
#wnd_frm_rp.head(10)


Total power in kW for operational turbines = 43600 kW 
Total power in kW for non-operational turbines = 17000 kW 


,rated_power_kw,operational,count
1,2300,True,13
3,2500,True,1
5,2800,True,4


### Question 1.b
How many turbines of each unique turbine model (based on rated power) are operational?

In [20]:
# Solution for Question 1.b
wnd_frm_rp[wnd_frm_rp['operational']==True]    

,rated_power_kw,operational,count
1,2300,True,13
3,2500,True,1
5,2800,True,4


### Question 1.c
In a single plot, display the turbine locations, their rated capacity and operational status.

In [42]:
# Solution for Question 1.c

# Import plotly and MinMaxScaler
import plotly.graph_objects as go
from sklearn.preprocessing import MinMaxScaler

# Define scaler used for making rated power changes appreciable
scaler = MinMaxScaler(feature_range=(10, 20))

# Define scattergeo plot parameters
fig = go.Figure(data=go.Scattergeo(
    lat=wnd_frm['lat'],lon=wnd_frm['lon'],
    text=wnd_frm['turbine_id']+' - '+wnd_frm['rated_power_kw'].astype(str)+'kW',
    marker=dict(size=scaler.fit_transform(gdf[['rated_power_kw']]),
        color=wnd_frm['operational'].astype(int),colorbar=dict(title="Operational State"),
        colorscale="PiYg"
    )
))

fig.update_layout(title='Turbine Locations (Hover over for locations & rated capacity)', geo=dict(fitbounds="locations", scope="usa"))

fig.show()